In [1]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

TECHNICAL_DEBT = "TECH_DEBT1 Self Admitted Technical Debt"
CODE_COMMENTS = "TECH_DEBT2 Commented Out Code"
MULTI_SERVICE = "SER2 Multi Service"
GOD_DI = "DI8 God Di Class"

ISSUE_TYPE = "issue_type"

STATIC_EVAL = "STATIC"
MACHINE_LEARNING_EVAL = "MACHINE_LEARNING"

FULLY_QUALIFIED_NAME = "fully_qualified_name"
GOD_DI_MANUAL = "god_di_manual"
MULTI_SERVICE_MANUAL = "multi_service_manual"

DETECTOR_TYPE = "detector_type"

In [2]:
# jead_result_file = input("Enter exported CSV file path: ")
jead_result_file = "/home/martin/Projects/jead-eval/klaw-issues/JoinedData-old-1.csv"
# manual_eval_file = input("Enter manual evaluation CSV file path: ")
manual_eval_file = "/home/martin/Projects/jead-eval/klaw-issues/manual-eval.csv"

result_pd = pd.read_csv(jead_result_file)
filtered_technical_debt = result_pd[result_pd[ISSUE_TYPE] == TECHNICAL_DEBT]
filtered_code_comments = result_pd[result_pd[ISSUE_TYPE] == CODE_COMMENTS]
filtered_multi_service = result_pd[result_pd[ISSUE_TYPE] == MULTI_SERVICE]
filtered_god_di = result_pd[result_pd[ISSUE_TYPE] == GOD_DI]

manual_eval = pd.read_csv(manual_eval_file)

god_di_true = manual_eval[manual_eval[GOD_DI_MANUAL] == "T"][FULLY_QUALIFIED_NAME]
god_di_false = manual_eval[manual_eval[GOD_DI_MANUAL] == "F"][FULLY_QUALIFIED_NAME]

true_positives = filtered_god_di[
    filtered_god_di[FULLY_QUALIFIED_NAME].isin(god_di_true)
]
true_positives_ml = true_positives[
    true_positives[DETECTOR_TYPE] == MACHINE_LEARNING_EVAL
]
true_positives_static = true_positives[true_positives[DETECTOR_TYPE] == STATIC_EVAL]

false_positives = filtered_god_di[
    filtered_god_di[FULLY_QUALIFIED_NAME].isin(god_di_false)
]
false_positives_ml = false_positives[
    false_positives[DETECTOR_TYPE] == MACHINE_LEARNING_EVAL
]
false_positives_static = false_positives[false_positives[DETECTOR_TYPE] == STATIC_EVAL]

true_negatives = god_di_false[~god_di_false.isin(filtered_god_di[FULLY_QUALIFIED_NAME])]
true_negatives_ml = [
    name
    for name in true_negatives
    if name
    not in filtered_god_di[filtered_god_di[DETECTOR_TYPE] == MACHINE_LEARNING_EVAL][
        FULLY_QUALIFIED_NAME
    ].tolist()
]
true_negatives_static = [
    name
    for name in true_negatives
    if name
    not in filtered_god_di[filtered_god_di[DETECTOR_TYPE] == STATIC_EVAL][
        FULLY_QUALIFIED_NAME
    ].tolist()
]

false_negatives = god_di_true[~god_di_true.isin(filtered_god_di[FULLY_QUALIFIED_NAME])]
false_negatives_ml = [
    name
    for name in false_negatives
    if name
    not in filtered_god_di[filtered_god_di[DETECTOR_TYPE] == MACHINE_LEARNING_EVAL][
        FULLY_QUALIFIED_NAME
    ].tolist()
]
false_negatives_static = [
    name
    for name in false_negatives
    if name
    not in filtered_god_di[filtered_god_di[DETECTOR_TYPE] == STATIC_EVAL][
        FULLY_QUALIFIED_NAME
    ].tolist()
]

In [3]:
# Prepare data for scikit-learn metrics
# Create y_true and y_pred for all samples in manual evaluation
all_names = manual_eval[FULLY_QUALIFIED_NAME].tolist()

# Overall metrics (combining ML and Static)
y_true_overall = [1 if name in god_di_true.tolist() else 0 for name in all_names]

# ML detector metrics
ml_detected_names = filtered_god_di[
    filtered_god_di[DETECTOR_TYPE] == MACHINE_LEARNING_EVAL
][FULLY_QUALIFIED_NAME].tolist()
y_pred_ml = [1 if name in ml_detected_names else 0 for name in all_names]

# Static detector metrics
static_detected_names = filtered_god_di[filtered_god_di[DETECTOR_TYPE] == STATIC_EVAL][
    FULLY_QUALIFIED_NAME
].tolist()
y_pred_static = [1 if name in static_detected_names else 0 for name in all_names]


# Compute metrics for ML detector
print("\n" + "=" * 60)
print("MACHINE LEARNING DETECTOR METRICS")
print("=" * 60)
print(f"Accuracy:  {accuracy_score(y_true_overall, y_pred_ml):.4f}")
print(f"Precision: {precision_score(y_true_overall, y_pred_ml, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_true_overall, y_pred_ml, zero_division=0):.4f}")
print(f"F1-Score:  {f1_score(y_true_overall, y_pred_ml, zero_division=0):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_true_overall, y_pred_ml))
print("\nClassification Report:")
print(
    classification_report(
        y_true_overall,
        y_pred_ml,
        target_names=["Not God DI", "God DI"],
        zero_division=0,
    )
)

# Compute metrics for Static detector
print("\n" + "=" * 60)
print("STATIC DETECTOR METRICS")
print("=" * 60)
print(f"Accuracy:  {accuracy_score(y_true_overall, y_pred_static):.4f}")
print(
    f"Precision: {precision_score(y_true_overall, y_pred_static, zero_division=0):.4f}"
)
print(f"Recall:    {recall_score(y_true_overall, y_pred_static, zero_division=0):.4f}")
print(f"F1-Score:  {f1_score(y_true_overall, y_pred_static, zero_division=0):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_true_overall, y_pred_static))
print("\nClassification Report:")
print(
    classification_report(
        y_true_overall,
        y_pred_static,
        target_names=["Not God DI", "God DI"],
        zero_division=0,
    )
)

# Summary counts
print("\n" + "=" * 60)
print("DETAILED COUNTS")
print("=" * 60)
print(f"True Positives (Overall): {len(true_positives)}")
print(f"  - ML: {len(true_positives_ml)}")
print(f"  - Static: {len(true_positives_static)}")
print(f"\nFalse Positives (Overall): {len(false_positives)}")
print(f"  - ML: {len(false_positives_ml)}")
print(f"  - Static: {len(false_positives_static)}")
print(f"\nTrue Negatives (Overall): {len(true_negatives)}")
print(f"  - ML: {len(true_negatives_ml)}")
print(f"  - Static: {len(true_negatives_static)}")
print(f"\nFalse Negatives (Overall): {len(false_negatives)}")
print(f"  - ML: {len(false_negatives_ml)}")
print(f"  - Static: {len(false_negatives_static)}")


MACHINE LEARNING DETECTOR METRICS
Accuracy:  0.8990
Precision: 0.5714
Recall:    0.3636
F1-Score:  0.4444

Confusion Matrix:
[[85  3]
 [ 7  4]]

Classification Report:
              precision    recall  f1-score   support

  Not God DI       0.92      0.97      0.94        88
      God DI       0.57      0.36      0.44        11

    accuracy                           0.90        99
   macro avg       0.75      0.66      0.69        99
weighted avg       0.88      0.90      0.89        99


STATIC DETECTOR METRICS
Accuracy:  0.9192
Precision: 1.0000
Recall:    0.2727
F1-Score:  0.4286

Confusion Matrix:
[[88  0]
 [ 8  3]]

Classification Report:
              precision    recall  f1-score   support

  Not God DI       0.92      1.00      0.96        88
      God DI       1.00      0.27      0.43        11

    accuracy                           0.92        99
   macro avg       0.96      0.64      0.69        99
weighted avg       0.93      0.92      0.90        99


DETAILED COUNTS
T